#**BLOQUE 0 — Introducción metodológica**

# **01_clean_salary.ipynb**

## **Ingeniería del dato — Fuente salarial**

### **Objetivo del notebook**
Este notebook documenta el proceso de extracción, limpieza, transformación, validación y tratamiento de valores faltantes de la fuente salarial del INE (Tabla 28201), con el fin de construir una tabla reproducible, homogénea y trazable para su integración posterior con el resto de fuentes del TFG.

### **Fuente utilizada**
- Instituto Nacional de Estadística (INE)
- Encuesta de Estructura Salarial
- [Tabla 28201](https://www.ine.es/jaxiT3/Tabla.htm?t=28201&L=0)

### **Relevancia de la fuente**
La fuente salarial constituye uno de los pilares del proyecto, ya que permite medir la capacidad de pago de la población joven y, posteriormente, calcular el esfuerzo salarial necesario para acceder a una vivienda en alquiler.

### **Problemas metodológicos de la fuente**
El archivo original no presenta una estructura tabular lista para análisis, sino una jerarquía orientada a la lectura (territorio → grupo de edad → año). Además, contiene valores faltantes y marcas de baja robustez muestral, por lo que requiere un proceso explícito de transformación y validación.

### **Salidas del notebook**
- `salary_observed.csv`: versión observada y limpia de la fuente
- `salary_completed.csv`: versión con valores faltantes imputados
- `salary_final.csv`: versión armonizada con estimaciones 2024–2025

## **Criterio metodológico aplicado**

De acuerdo con el planteamiento metodológico del anteproyecto, la construcción de la serie salarial se basa en:
- la selección de la Tabla 28201 del INE,
- la transformación de la fuente original a formato largo,
- el tratamiento de valores faltantes mediante interpolación lineal en huecos cortos,
- la imputación mediante tasa de crecimiento nacional en huecos largos,
- y la estimación posterior de 2024 y 2025 para armonizar el panel final.

Este notebook sigue una lógica de ingeniería del dato estructurada en fases: extracción, inspección de la fuente, limpieza estructural, validación, tratamiento de nulos, transformaciones y generación de salidas finales.

## **Problema técnico de la fuente**

La fuente salarial original del INE no presenta una estructura tabular lista para análisis, sino una jerarquía orientada a la lectura:
- territorio
- grupo de edad
- año

Además, contiene valores especiales que requieren tratamiento explícito:
- `..` para datos no facilitados por insuficiencia muestral
- valores con signo negativo, que no representan salarios negativos reales, sino observaciones con baja robustez muestral

Por ello, el notebook no se limita a leer el archivo, sino que reconstruye su lógica interna y documenta todas las transformaciones aplicadas.

# **BLOQUE 1 — Extracción e inspección de la fuente**

In [146]:
# ============================================================
# BLOQUE 1. EXTRACCIÓN E INSPECCIÓN DE LA FUENTE
# 1.1. ENTORNO DE TRABAJO Y LIBRERÍAS
# En este bloque montamos Google Drive e importamos las
# librerías necesarias para leer e inspeccionar la fuente.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from pathlib import Path

# Opciones de visualización para trabajar más cómodamente
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [147]:
# ============================================================
# 1.2. DEFINICIÓN DE RUTAS
# Definimos la ruta del Excel bruto y las rutas de salida
# que usaremos más adelante en el notebook.
# ============================================================

salary_file = "/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data raw/INE_Ganancia Media Mensual por Trabajador (2010-2023).xlsx"

output_observed = "/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_observed.csv"
output_completed = "/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_completed.csv"
output_final = "/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_final.csv"

print("Archivo de entrada:")
print(salary_file)

print("\nArchivos de salida previstos:")
print(output_observed)
print(output_completed)
print(output_final)

Archivo de entrada:
/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data raw/INE_Ganancia Media Mensual por Trabajador (2010-2023).xlsx

Archivos de salida previstos:
/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_observed.csv
/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_completed.csv
/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_final.csv


## **Lectura inicial de la fuente**

En este primer bloque no transformamos todavía los datos. El objetivo es leer el archivo bruto, comprobar que la hoja es correcta y entender visualmente la estructura original de la fuente.

In [148]:
# ============================================================
# 1.3. LECTURA DEL EXCEL BRUTO
# Leemos la hoja completa sin asumir encabezados limpios,
# ya que la estructura original está orientada a lectura.
# ============================================================

df_raw = pd.read_excel(
    salary_file,
    sheet_name="tabla-28201",
    header=None)

print("Dimensiones del Excel bruto:", df_raw.shape)
display(df_raw.head(25))

Dimensiones del Excel bruto: (1654, 4)


,0,1,2,3
0,Resultados nacionales y por comunidades aútonomas,NaN,NaN,NaN
1,Ganancia media anual por trabajador,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,Sexo y edad,NaN,NaN,NaN
4,Unidades: €,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN
6,,Ambos sexos,Mujeres,Hombres
7,Total Nacional,NaN,NaN,NaN
8,Todas las edades,NaN,NaN,NaN
9,2023,"28,049.94","25,591.31","30,372.49"


## **Inspección inicial de la estructura**

Antes de limpiar o transformar la fuente, es necesario localizar:
- el comienzo real del bloque de observaciones,
- el final del bloque útil,
- y la forma exacta en que el INE organiza territorio, grupo de edad y año.

Este paso es clave porque permite justificar después por qué se recorta el archivo y cómo se reconstruye su jerarquía interna.

In [149]:
# ============================================================
# 1.4. LOCALIZAR EL BLOQUE ÚTIL DE DATOS
# Buscamos:
# - la primera fila de datos: 'Total Nacional'
# - la fila donde empiezan las notas metodológicas: 'Notas:'
# ============================================================

start_idx = df_raw[df_raw[0].astype(str).str.strip() == "Total Nacional"].index.min()
end_idx = df_raw[df_raw[0].astype(str).str.contains("Notas:", na=False)].index.min()

print("Fila donde empieza 'Total Nacional':", start_idx)
print("Fila donde empieza 'Notas:':", end_idx)

Fila donde empieza 'Total Nacional': 7
Fila donde empieza 'Notas:': 1647


In [150]:
# ============================================================
# 1.5. INSPECCIÓN VISUAL DE LA FUENTE
# Mostramos un tramo amplio del inicio y del final para
# entender la organización real del Excel.
# ============================================================

print("Primeras 60 filas del Excel bruto:")
display(df_raw.head(60))

print("\nÚltimas 20 filas del Excel bruto:")
display(df_raw.tail(20))

Primeras 60 filas del Excel bruto:


,0,1,2,3
0,Resultados nacionales y por comunidades aútonomas,NaN,NaN,NaN
1,Ganancia media anual por trabajador,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,Sexo y edad,NaN,NaN,NaN
4,Unidades: €,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN
6,,Ambos sexos,Mujeres,Hombres
7,Total Nacional,NaN,NaN,NaN
8,Todas las edades,NaN,NaN,NaN
9,2023,"28,049.94","25,591.31","30,372.49"



Últimas 20 filas del Excel bruto:


,0,1,2,3
1634,2020,"24,465.45","-21,325.96","28,049.39"
1635,2019,"24,630.95","-20,795.89","28,986.96"
1636,2018,"25,014.42","-21,174.15","28,976.41"
1637,2017,"24,147.76","-21,128.20","26,780.45"
1638,2016,"24,151.12","-20,880.48","27,056.47"
1639,2015,"24,730.65","-20,850.75","27,642.02"
1640,2014,"25,604.47","-20,739.28","-28,897.03"
1641,2013,"23,332.74","-21,816.87","24,350.32"
1642,2012,"23,494.49","-20,597.19","-25,361.70"
1643,2011,"25,500.59",-21576,"-28,084.19"


## **Conclusión del bloque 1**

Tras la inspección inicial, se confirma que:
- la hoja de trabajo correcta es `tabla-28201`,
- la estructura no es tabular sino jerárquica,
- el bloque útil comienza en `Total Nacional`,
- y finaliza antes de `Notas:`.

En consecuencia, el siguiente paso será recortar el bloque útil y reconstruir la jerarquía original de la fuente para convertirla a formato largo.

# **BLOQUE 2 — Limpieza estructural y transformación a formato largo**



Una vez identificado el bloque útil de observaciones, el siguiente paso consiste en reconstruir la estructura lógica de la fuente. El Excel original del INE no está diseñado para análisis directo, sino para lectura humana. Por ello, es necesario:

- recortar únicamente las filas con datos útiles,
- eliminar filas vacías,
- reconstruir la jerarquía territorio → grupo de edad → año,
- convertir la fuente a formato largo,
- y tratar los valores especiales que aparecen en la tabla.

Este bloque transforma la fuente bruta en una estructura analítica reproducible.

In [151]:
# ============================================================
# BLOQUE 2. LIMPIEZA ESTRUCTURAL Y TRANSFORMACIÓN
# 2.1. RECORTE DEL BLOQUE ÚTIL
# Nos quedamos únicamente con el bloque de observaciones,
# desde 'Total Nacional' hasta justo antes de 'Notas:'.
# ============================================================

df_block = df_raw.loc[start_idx:end_idx - 1].copy()

# Nos quedamos con las cuatro columnas relevantes
df_block = df_block.iloc[:, :4].copy()
df_block.columns = ["raw_label", "both_sexes", "women", "men"]

print("Dimensiones del bloque útil:", df_block.shape)
display(df_block.head(20))
display(df_block.tail(20))

Dimensiones del bloque útil: (1640, 4)


,raw_label,both_sexes,women,men
7,Total Nacional,NaN,NaN,NaN
8,Todas las edades,NaN,NaN,NaN
9,2023,"28,049.94","25,591.31","30,372.49"
10,2022,"26,948.87","24,359.82","29,381.84"
11,2021,"25,896.82","23,175.95","28,388.69"
12,2020,"25,165.51","22,467.48","27,642.52"
13,2019,"24,395.98","21,682.02","26,934.38"
14,2018,"24,009.12","21,011.89","26,738.19"
15,2017,"23,646.50","20,607.85","26,391.84"
16,2016,"23,156.34","20,131.41","25,924.43"


,raw_label,both_sexes,women,men
1627,2012,"23,803.80","-20,108.59","27,372.20"
1628,2011,"24,828.46","-22,583.68","26,679.15"
1629,2010,"23,001.51","-19,600.41","26,441.72"
1630,55 y más años,NaN,NaN,NaN
1631,2023,"29,360.28","26,293.86","32,599.76"
1632,2022,"27,221.82","24,334.42","30,287.33"
1633,2021,"25,047.12","21,554.70","29,415.48"
1634,2020,"24,465.45","-21,325.96","28,049.39"
1635,2019,"24,630.95","-20,795.89","28,986.96"
1636,2018,"25,014.42","-21,174.15","28,976.41"


## **Limpieza estructural inicial**

En esta fase se eliminan filas vacías y se normaliza la columna principal (`raw_label`) para poder recorrer la fuente con seguridad. Este paso es necesario porque el bloque útil todavía contiene espacios, filas vacías y elementos no analíticos.

In [152]:
# ============================================================
# 2.2. LIMPIEZA ESTRUCTURAL BÁSICA
# Eliminamos filas vacías y normalizamos la etiqueta principal.
# ============================================================

# Convertimos cadenas vacías en NaN
df_block["raw_label"] = df_block["raw_label"].replace(r"^\s*$", np.nan, regex=True)

# Eliminamos filas donde raw_label esté vacío
df_block = df_block[df_block["raw_label"].notna()].copy()

# Convertimos a texto y limpiamos espacios
df_block["raw_label"] = df_block["raw_label"].astype(str).str.strip()

# Eliminamos valores basura si apareciesen como texto
df_block = df_block[
    ~df_block["raw_label"].str.lower().isin(["nan", "none", ""])
].copy()

# Reseteamos índice para trabajar cómodamente
df_block = df_block.reset_index(drop=True)

print("Dimensiones tras limpieza estructural:", df_block.shape)
display(df_block.head(15))
display(df_block.tail(15))

Dimensiones tras limpieza estructural: (1638, 4)


,raw_label,both_sexes,women,men
0,Total Nacional,NaN,NaN,NaN
1,Todas las edades,NaN,NaN,NaN
2,2023,"28,049.94","25,591.31","30,372.49"
3,2022,"26,948.87","24,359.82","29,381.84"
4,2021,"25,896.82","23,175.95","28,388.69"
5,2020,"25,165.51","22,467.48","27,642.52"
6,2019,"24,395.98","21,682.02","26,934.38"
7,2018,"24,009.12","21,011.89","26,738.19"
8,2017,"23,646.50","20,607.85","26,391.84"
9,2016,"23,156.34","20,131.41","25,924.43"


,raw_label,both_sexes,women,men
1623,55 y más años,NaN,NaN,NaN
1624,2023,"29,360.28","26,293.86","32,599.76"
1625,2022,"27,221.82","24,334.42","30,287.33"
1626,2021,"25,047.12","21,554.70","29,415.48"
1627,2020,"24,465.45","-21,325.96","28,049.39"
1628,2019,"24,630.95","-20,795.89","28,986.96"
1629,2018,"25,014.42","-21,174.15","28,976.41"
1630,2017,"24,147.76","-21,128.20","26,780.45"
1631,2016,"24,151.12","-20,880.48","27,056.47"
1632,2015,"24,730.65","-20,850.75","27,642.02"


In [153]:
# ============================================================
# 2.3. INSPECCIÓN VISUAL DE LA JERARQUÍA
# Comprobamos cómo aparece la estructura real del bloque ya limpio.
# ============================================================

for i, value in enumerate(df_block["raw_label"].head(50)):
    print(i, repr(value))

0 'Total Nacional'
1 'Todas las edades'
2 '2023'
3 '2022'
4 '2021'
5 '2020'
6 '2019'
7 '2018'
8 '2017'
9 '2016'
10 '2015'
11 '2014'
12 '2013'
13 '2012'
14 '2011'
15 '2010'
16 'Menos de 25 años'
17 '2023'
18 '2022'
19 '2021'
20 '2020'
21 '2019'
22 '2018'
23 '2017'
24 '2016'
25 '2015'
26 '2014'
27 '2013'
28 '2012'
29 '2011'
30 '2010'
31 'De 25 a 34 años'
32 '2023'
33 '2022'
34 '2021'
35 '2020'
36 '2019'
37 '2018'
38 '2017'
39 '2016'
40 '2015'
41 '2014'
42 '2013'
43 '2012'
44 '2011'
45 '2010'
46 'De 35 a 44 años'
47 '2023'
48 '2022'
49 '2021'


## **Reconstrucción de la jerarquía**

La fuente sigue una lógica jerárquica:
- primero aparece el territorio,
- después el grupo de edad,
- y después los años con sus tres columnas salariales.

Para poder trabajar con ella, debemos recorrer fila a fila la tabla y reconstruir esa lógica de forma explícita.

In [154]:
# ============================================================
# 2.4. DEFINIR LOS GRUPOS DE EDAD ESPERADOS
# Esto permite distinguir grupos de edad frente a territorios.
# ============================================================

age_groups_ine = {
    "Todas las edades",
    "Menos de 25 años",
    "De 25 a 34 años",
    "De 35 a 44 años",
    "De 45 a 54 años",
    "55 y más años",
}

print("Grupos de edad detectables:")
for age in sorted(age_groups_ine):
    print("-", age)

Grupos de edad detectables:
- 55 y más años
- De 25 a 34 años
- De 35 a 44 años
- De 45 a 54 años
- Menos de 25 años
- Todas las edades


In [155]:
# ============================================================
# 2.5. FUNCIÓN AUXILIAR PARA DETECTAR AÑOS
# Un año válido para esta fuente será un entero entre 2010 y 2023.
# ============================================================

def is_valid_year(value):
    """
    Devuelve True si el valor representa un año entre 2010 y 2023.
    """
    try:
        year = int(float(value))
        return 2010 <= year <= 2023
    except:
        return False

# Comprobaciones rápidas
print(is_valid_year("2023"))            # True
print(is_valid_year("Total Nacional"))  # False
print(is_valid_year("55 y más años"))   # False

True
False
False


In [156]:
# ============================================================
# 2.6. RECONSTRUIR LA JERARQUÍA Y CREAR LA TABLA LARGA
# Recorremos fila a fila:
# - si la fila es territorio, actualizamos territorio
# - si la fila es grupo de edad, actualizamos grupo de edad
# - si la fila es año, creamos 3 filas:
#   Total, Mujeres, Hombres
# ============================================================

records = []

current_territory = None
current_age_group = None

for _, row in df_block.iterrows():
    label = row["raw_label"]

    # Caso 1: grupo de edad
    if label in age_groups_ine:
        current_age_group = label
        continue

    # Caso 2: año
    if is_valid_year(label):
        year = int(float(label))

        if current_territory is not None and current_age_group is not None:
            records.append({
                "territory_name": current_territory,
                "age_group": current_age_group,
                "year": year,
                "sex": "Total",
                "salary_raw": row["both_sexes"]
            })
            records.append({
                "territory_name": current_territory,
                "age_group": current_age_group,
                "year": year,
                "sex": "Mujeres",
                "salary_raw": row["women"]
            })
            records.append({
                "territory_name": current_territory,
                "age_group": current_age_group,
                "year": year,
                "sex": "Hombres",
                "salary_raw": row["men"]
            })
        continue

    # Caso 3: territorio
    current_territory = label
    current_age_group = None

# Convertimos la lista a DataFrame
df_long = pd.DataFrame(records)

print("Dimensiones de la tabla larga:", df_long.shape)
display(df_long.head(20))
display(df_long.tail(20))

Dimensiones de la tabla larga: (4536, 5)


,territory_name,age_group,year,sex,salary_raw
0,Total Nacional,Todas las edades,2023,Total,"28,049.94"
1,Total Nacional,Todas las edades,2023,Mujeres,"25,591.31"
2,Total Nacional,Todas las edades,2023,Hombres,"30,372.49"
3,Total Nacional,Todas las edades,2022,Total,"26,948.87"
4,Total Nacional,Todas las edades,2022,Mujeres,"24,359.82"
5,Total Nacional,Todas las edades,2022,Hombres,"29,381.84"
6,Total Nacional,Todas las edades,2021,Total,"25,896.82"
7,Total Nacional,Todas las edades,2021,Mujeres,"23,175.95"
8,Total Nacional,Todas las edades,2021,Hombres,"28,388.69"
9,Total Nacional,Todas las edades,2020,Total,"25,165.51"


,territory_name,age_group,year,sex,salary_raw
4516,"Rioja, La",55 y más años,2016,Mujeres,"-20,880.48"
4517,"Rioja, La",55 y más años,2016,Hombres,"27,056.47"
4518,"Rioja, La",55 y más años,2015,Total,"24,730.65"
4519,"Rioja, La",55 y más años,2015,Mujeres,"-20,850.75"
4520,"Rioja, La",55 y más años,2015,Hombres,"27,642.02"
4521,"Rioja, La",55 y más años,2014,Total,"25,604.47"
4522,"Rioja, La",55 y más años,2014,Mujeres,"-20,739.28"
4523,"Rioja, La",55 y más años,2014,Hombres,"-28,897.03"
4524,"Rioja, La",55 y más años,2013,Total,"23,332.74"
4525,"Rioja, La",55 y más años,2013,Mujeres,"-21,816.87"


In [157]:
# ============================================================
# 2.7. COMPROBACIONES BÁSICAS DE COHERENCIA
# ============================================================

print("Número de filas:", len(df_long))
print("Territorios únicos:", df_long["territory_name"].nunique())
print("Grupos de edad únicos:", sorted(df_long["age_group"].unique()))
print("Sexos únicos:", sorted(df_long["sex"].unique()))
print("Año mínimo:", df_long["year"].min())
print("Año máximo:", df_long["year"].max())

expected_rows = 18 * 6 * 14 * 3
print("Filas esperadas:", expected_rows)

Número de filas: 4536
Territorios únicos: 18
Grupos de edad únicos: ['55 y más años', 'De 25 a 34 años', 'De 35 a 44 años', 'De 45 a 54 años', 'Menos de 25 años', 'Todas las edades']
Sexos únicos: ['Hombres', 'Mujeres', 'Total']
Año mínimo: 2010
Año máximo: 2023
Filas esperadas: 4536


## **Tratamiento de valores especiales**

La tabla del INE contiene dos tipos de valores especiales:

- `..` indica que el dato no se facilita por insuficiencia muestral.
- algunos valores aparecen con signo negativo; estos no representan salarios negativos reales, sino observaciones marcadas por el INE con menor robustez muestral.

Por tanto:
- `..` se tratará como faltante,
- los valores negativos se convertirán a valor absoluto,
- y se conservará una bandera específica de baja muestra para no perder esa información.

In [158]:
# ============================================================
# 2.8. FUNCIÓN DE TRATAMIENTO DE VALORES ESPECIALES
  # Reglas del INE:
    # - '..'  => dato no facilitado por muestra < 100 -> faltante
    # - signo negativo delante => dato con muestra entre 100 y 500
    #   y alta variabilidad. NO es salario negativo real.
# ============================================================

def parse_salary_value(value):
    """
    Devuelve:
    - salary_annual_eur
    - salary_missing_flag
    - salary_low_sample_flag
    """
    # Caso 1: NaN real
    if pd.isna(value):
        return np.nan, 1, 0

    value_str = str(value).strip()

    # Caso 2: faltante explícito del INE
    if value_str == "..":
        return np.nan, 1, 0

    # Caso 3: convertir a número
    try:
        numeric_value = float(value_str)

        # Si viene negativo, tomamos valor absoluto
        # y marcamos baja robustez muestral
        if numeric_value < 0:
            return abs(numeric_value), 0, 1

        return numeric_value, 0, 0

    except:
        return np.nan, 1, 0

In [159]:
# ============================================================
# 2.9. APLICAR EL TRATAMIENTO DE VALORES ESPECIALES
# ============================================================

parsed_values = df_long["salary_raw"].apply(parse_salary_value)

df_long["salary_annual_eur"] = parsed_values.apply(lambda x: x[0])
df_long["salary_missing_flag"] = parsed_values.apply(lambda x: x[1])
df_long["salary_low_sample_flag"] = parsed_values.apply(lambda x: x[2])

display(df_long.head(20))

,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
0,Total Nacional,Todas las edades,2023,Total,"28,049.94","28,049.94",0,0
1,Total Nacional,Todas las edades,2023,Mujeres,"25,591.31","25,591.31",0,0
2,Total Nacional,Todas las edades,2023,Hombres,"30,372.49","30,372.49",0,0
3,Total Nacional,Todas las edades,2022,Total,"26,948.87","26,948.87",0,0
4,Total Nacional,Todas las edades,2022,Mujeres,"24,359.82","24,359.82",0,0
5,Total Nacional,Todas las edades,2022,Hombres,"29,381.84","29,381.84",0,0
6,Total Nacional,Todas las edades,2021,Total,"25,896.82","25,896.82",0,0
7,Total Nacional,Todas las edades,2021,Mujeres,"23,175.95","23,175.95",0,0
8,Total Nacional,Todas las edades,2021,Hombres,"28,388.69","28,388.69",0,0
9,Total Nacional,Todas las edades,2020,Total,"25,165.51","25,165.51",0,0


In [160]:
# ============================================================
# TEST_01. ASEGURARSE DE QUE NO QUEDAN SALARIOS NEGATIVOS
# ============================================================

negative_salaries = df_long[df_long["salary_annual_eur"] < 0]

print("Número de salarios negativos después de limpiar:", len(negative_salaries))
display(negative_salaries.head())

Número de salarios negativos después de limpiar: 0


,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag


In [161]:
# ============================================================
# TEST_02. COMPROBAR QUE LOS NEGATIVOS ORIGINALES AHORA
# SON POSITIVOS Y ESTÁN MARCADOS COMO BAJA MUESTRA
# ============================================================

# Filas donde el valor bruto era negativo
neg_raw = df_long[pd.to_numeric(df_long["salary_raw"], errors="coerce") < 0].copy()

print("Número de valores brutos negativos:", len(neg_raw))

display(
    neg_raw[
        ["territory_name", "age_group", "year", "sex", "salary_raw", "salary_annual_eur", "salary_missing_flag", "salary_low_sample_flag"]
    ].head(15)
)

Número de valores brutos negativos: 607


,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
295,Andalucía,Menos de 25 años,2023,Mujeres,"-12,785.10","12,785.10",0,1
298,Andalucía,Menos de 25 años,2022,Mujeres,"-12,055.16","12,055.16",0,1
299,Andalucía,Menos de 25 años,2022,Hombres,"-16,443.75","16,443.75",0,1
301,Andalucía,Menos de 25 años,2021,Mujeres,"-11,646.17","11,646.17",0,1
302,Andalucía,Menos de 25 años,2021,Hombres,"-13,389.36","13,389.36",0,1
303,Andalucía,Menos de 25 años,2020,Total,"-11,313.88","11,313.88",0,1
304,Andalucía,Menos de 25 años,2020,Mujeres,"-10,674.17","10,674.17",0,1
305,Andalucía,Menos de 25 años,2020,Hombres,"-11,920.68","11,920.68",0,1
307,Andalucía,Menos de 25 años,2019,Mujeres,"-10,615.13","10,615.13",0,1
308,Andalucía,Menos de 25 años,2019,Hombres,"-12,772.33","12,772.33",0,1


In [162]:
# ============================================================
# TEST_03. COMPROBAR QUE LOS '..' SE HAN CONVERTIDO EN NULO
# ============================================================

dots_rows = df_long[df_long["salary_raw"].astype(str).str.strip() == ".."].copy()

print("Número de filas con '..' en salary_raw:", len(dots_rows))

display(
    dots_rows[
        ["territory_name", "age_group", "year", "sex", "salary_raw", "salary_annual_eur", "salary_missing_flag"]
    ].head(15)
)

Número de filas con '..' en salary_raw: 163


,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag
553,Aragón,Menos de 25 años,2021,Mujeres,..,NaN,1
556,Aragón,Menos de 25 años,2020,Mujeres,..,NaN,1
559,Aragón,Menos de 25 años,2019,Mujeres,..,NaN,1
565,Aragón,Menos de 25 años,2017,Mujeres,..,NaN,1
568,Aragón,Menos de 25 años,2016,Mujeres,..,NaN,1
569,Aragón,Menos de 25 años,2016,Hombres,..,NaN,1
571,Aragón,Menos de 25 años,2015,Mujeres,..,NaN,1
577,Aragón,Menos de 25 años,2013,Mujeres,..,NaN,1
580,Aragón,Menos de 25 años,2012,Mujeres,..,NaN,1
799,"Asturias, Principado de",Menos de 25 años,2023,Mujeres,..,NaN,1


In [163]:
# ============================================================
# TEST_04. COMPROBACIÓN MANUAL DE UNA FILA CONCRETA
# Cambia los valores por un caso que hayas visto en el Excel.
# ============================================================

example = df_long[
    (df_long["territory_name"] == "Aragón") &
    (df_long["age_group"] == "Menos de 25 años") &
    (df_long["year"] == 2012) &
    (df_long["sex"] == "Total")
]

display(example)

,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
579,Aragón,Menos de 25 años,2012,Total,"-11,156.57","11,156.57",0,1


In [164]:
# ============================================================
# 2.10. VALIDACIÓN DEL TRATAMIENTO DE VALORES ESPECIALES
# ============================================================

print("Nulos en salary_annual_eur:", int(df_long["salary_annual_eur"].isna().sum()))
print("Missing flag = 1:", int(df_long["salary_missing_flag"].sum()))
print("Low sample flag = 1:", int(df_long["salary_low_sample_flag"].sum()))
print("Salarios negativos tras limpieza:", int((df_long["salary_annual_eur"] < 0).sum()))

print("\nEjemplos de datos faltantes:")
display(
    df_long[df_long["salary_missing_flag"] == 1][
        ["territory_name", "age_group", "year", "sex", "salary_raw",
         "salary_annual_eur", "salary_missing_flag", "salary_low_sample_flag"]
    ].head(10)
)

print("\nEjemplos de baja muestra:")
display(
    df_long[df_long["salary_low_sample_flag"] == 1][
        ["territory_name", "age_group", "year", "sex", "salary_raw",
         "salary_annual_eur", "salary_missing_flag", "salary_low_sample_flag"]
    ].head(10)
)

Nulos en salary_annual_eur: 163
Missing flag = 1: 163
Low sample flag = 1: 607
Salarios negativos tras limpieza: 0

Ejemplos de datos faltantes:


,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
553,Aragón,Menos de 25 años,2021,Mujeres,..,NaN,1,0
556,Aragón,Menos de 25 años,2020,Mujeres,..,NaN,1,0
559,Aragón,Menos de 25 años,2019,Mujeres,..,NaN,1,0
565,Aragón,Menos de 25 años,2017,Mujeres,..,NaN,1,0
568,Aragón,Menos de 25 años,2016,Mujeres,..,NaN,1,0
569,Aragón,Menos de 25 años,2016,Hombres,..,NaN,1,0
571,Aragón,Menos de 25 años,2015,Mujeres,..,NaN,1,0
577,Aragón,Menos de 25 años,2013,Mujeres,..,NaN,1,0
580,Aragón,Menos de 25 años,2012,Mujeres,..,NaN,1,0
799,"Asturias, Principado de",Menos de 25 años,2023,Mujeres,..,NaN,1,0



Ejemplos de baja muestra:


,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
295,Andalucía,Menos de 25 años,2023,Mujeres,"-12,785.10","12,785.10",0,1
298,Andalucía,Menos de 25 años,2022,Mujeres,"-12,055.16","12,055.16",0,1
299,Andalucía,Menos de 25 años,2022,Hombres,"-16,443.75","16,443.75",0,1
301,Andalucía,Menos de 25 años,2021,Mujeres,"-11,646.17","11,646.17",0,1
302,Andalucía,Menos de 25 años,2021,Hombres,"-13,389.36","13,389.36",0,1
303,Andalucía,Menos de 25 años,2020,Total,"-11,313.88","11,313.88",0,1
304,Andalucía,Menos de 25 años,2020,Mujeres,"-10,674.17","10,674.17",0,1
305,Andalucía,Menos de 25 años,2020,Hombres,"-11,920.68","11,920.68",0,1
307,Andalucía,Menos de 25 años,2019,Mujeres,"-10,615.13","10,615.13",0,1
308,Andalucía,Menos de 25 años,2019,Hombres,"-12,772.33","12,772.33",0,1


## **Conclusión del bloque 2**

Tras este bloque, la fuente salarial ya ha sido:
- recortada,
- limpiada estructuralmente,
- reconstruida a formato largo,
- y tratada en sus valores especiales.

La siguiente fase consistirá en normalizar etiquetas, validar la tabla observada y guardar la primera salida oficial del notebook: `salary_observed.csv`.

# **BLOQUE 3 — Normalización y validación de la versión observada**



En este bloque se normalizan las etiquetas necesarias para facilitar la integración posterior con otras fuentes del proyecto. Además, se valida la coherencia interna de la tabla observada antes de guardarla como salida oficial.

La validación incluye:
- cobertura territorial,
- rango temporal,
- sexos y grupos de edad,
- ausencia de duplicados,
- y comprobaciones básicas de calidad del dato.

In [165]:
# ============================================================
# BLOQUE 3. NORMALIZACIÓN Y VALIDACIÓN
# 3.1. NORMALIZAR NOMBRES DE TERRITORIOS
# Dejamos nombres homogéneos para facilitar merges futuros.
# ============================================================

territory_map = {
    "Total Nacional": "España",
    "Andalucía": "Andalucía",
    "Aragón": "Aragón",
    "Asturias, Principado de": "Asturias",
    "Balears, Illes": "Baleares",
    "Canarias": "Canarias",
    "Cantabria": "Cantabria",
    "Castilla y León": "Castilla y León",
    "Castilla - La Mancha": "Castilla-La Mancha",
    "Cataluña": "Cataluña",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Extremadura": "Extremadura",
    "Galicia": "Galicia",
    "Madrid, Comunidad de": "Madrid",
    "Murcia, Región de": "Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "País Vasco": "País Vasco",
    "Rioja, La": "La Rioja",
}

df_long["territory_name"] = df_long["territory_name"].replace(territory_map)

print("Territorios normalizados:")
print(sorted(df_long["territory_name"].unique()))
print("\nNúmero de territorios:", df_long["territory_name"].nunique())

Territorios normalizados:
['Andalucía', 'Aragón', 'Asturias', 'Baleares', 'Canarias', 'Cantabria', 'Castilla y León', 'Castilla-La Mancha', 'Cataluña', 'Comunidad Valenciana', 'España', 'Extremadura', 'Galicia', 'La Rioja', 'Madrid', 'Murcia', 'Navarra', 'País Vasco']

Número de territorios: 18


In [166]:
# ============================================================
# 3.2. CREAR EL TIPO DE TERRITORIO
# Distinguimos España del resto de comunidades autónomas.
# ============================================================

df_long["territory_type"] = np.where(
    df_long["territory_name"] == "España",
    "España",
    "CCAA"
)

display(
    df_long[["territory_name", "territory_type"]]
    .drop_duplicates()
    .sort_values(["territory_type", "territory_name"])
)

,territory_name,territory_type
252,Andalucía,CCAA
504,Aragón,CCAA
756,Asturias,CCAA
1008,Baleares,CCAA
1260,Canarias,CCAA
1512,Cantabria,CCAA
1764,Castilla y León,CCAA
2016,Castilla-La Mancha,CCAA
2268,Cataluña,CCAA
2520,Comunidad Valenciana,CCAA


In [167]:
# ============================================================
# 3.3. NORMALIZAR GRUPOS DE EDAD
# Creamos etiquetas cortas, consistentes y útiles para análisis.
# ============================================================

age_group_map = {
    "Todas las edades": "all_ages",
    "Menos de 25 años": "lt_25",
    "De 25 a 34 años": "25_34",
    "De 35 a 44 años": "35_44",
    "De 45 a 54 años": "45_54",
    "55 y más años": "55_plus",
}

df_long["age_group"] = df_long["age_group"].replace(age_group_map)

print("Grupos de edad normalizados:")
print(sorted(df_long["age_group"].unique()))

Grupos de edad normalizados:
['25_34', '35_44', '45_54', '55_plus', 'all_ages', 'lt_25']


In [168]:
# ============================================================
# 3.4. CONSTRUIR LA TABLA OBSERVADA FINAL
# Dejamos únicamente las columnas definitivas de la versión
# observada y añadimos la trazabilidad de fuente.
# ============================================================

df_salary = df_long.copy()

df_salary["salary_source"] = "INE_EES_28201"

df_salary = df_salary[
    [
        "territory_name",
        "territory_type",
        "year",
        "sex",
        "age_group",
        "salary_annual_eur",
        "salary_missing_flag",
        "salary_low_sample_flag",
        "salary_source",
    ]
].copy()

print("Dimensiones de la tabla observada:", df_salary.shape)
display(df_salary.head(15))
display(df_salary.tail(15))

Dimensiones de la tabla observada: (4536, 9)


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source
0,España,España,2023,Total,all_ages,"28,049.94",0,0,INE_EES_28201
1,España,España,2023,Mujeres,all_ages,"25,591.31",0,0,INE_EES_28201
2,España,España,2023,Hombres,all_ages,"30,372.49",0,0,INE_EES_28201
3,España,España,2022,Total,all_ages,"26,948.87",0,0,INE_EES_28201
4,España,España,2022,Mujeres,all_ages,"24,359.82",0,0,INE_EES_28201
5,España,España,2022,Hombres,all_ages,"29,381.84",0,0,INE_EES_28201
6,España,España,2021,Total,all_ages,"25,896.82",0,0,INE_EES_28201
7,España,España,2021,Mujeres,all_ages,"23,175.95",0,0,INE_EES_28201
8,España,España,2021,Hombres,all_ages,"28,388.69",0,0,INE_EES_28201
9,España,España,2020,Total,all_ages,"25,165.51",0,0,INE_EES_28201


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source
4521,La Rioja,CCAA,2014,Total,55_plus,"25,604.47",0,0,INE_EES_28201
4522,La Rioja,CCAA,2014,Mujeres,55_plus,"20,739.28",0,1,INE_EES_28201
4523,La Rioja,CCAA,2014,Hombres,55_plus,"28,897.03",0,1,INE_EES_28201
4524,La Rioja,CCAA,2013,Total,55_plus,"23,332.74",0,0,INE_EES_28201
4525,La Rioja,CCAA,2013,Mujeres,55_plus,"21,816.87",0,1,INE_EES_28201
4526,La Rioja,CCAA,2013,Hombres,55_plus,"24,350.32",0,0,INE_EES_28201
4527,La Rioja,CCAA,2012,Total,55_plus,"23,494.49",0,0,INE_EES_28201
4528,La Rioja,CCAA,2012,Mujeres,55_plus,"20,597.19",0,1,INE_EES_28201
4529,La Rioja,CCAA,2012,Hombres,55_plus,"25,361.70",0,1,INE_EES_28201
4530,La Rioja,CCAA,2011,Total,55_plus,"25,500.59",0,0,INE_EES_28201


## **Validación de la tabla observada**

Antes de guardar la salida observada, se comprueba que la tabla tenga:
- la cobertura esperada,
- el rango temporal correcto,
- una estructura sin duplicados,
- y consistencia en las variables clave.

Este paso es importante porque evita arrastrar errores al resto del pipeline.

In [169]:
# ============================================================
# 3.5. ORDENAR Y VALIDAR ESTRUCTURA GENERAL
# ============================================================

df_salary = df_salary.sort_values(
    by=["territory_type", "territory_name", "age_group", "year", "sex"]
).reset_index(drop=True)

print("Número de filas:", len(df_salary))
print("Territorios únicos:", df_salary["territory_name"].nunique())
print("Sexos únicos:", sorted(df_salary["sex"].unique()))
print("Grupos de edad:", sorted(df_salary["age_group"].unique()))
print("Rango temporal:", df_salary["year"].min(), "-", df_salary["year"].max())

display(df_salary.head(20))

Número de filas: 4536
Territorios únicos: 18
Sexos únicos: ['Hombres', 'Mujeres', 'Total']
Grupos de edad: ['25_34', '35_44', '45_54', '55_plus', 'all_ages', 'lt_25']
Rango temporal: 2010 - 2023


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source
0,Andalucía,CCAA,2010,Hombres,25_34,"19,574.29",0,0,INE_EES_28201
1,Andalucía,CCAA,2010,Mujeres,25_34,"15,130.18",0,0,INE_EES_28201
2,Andalucía,CCAA,2010,Total,25_34,"17,299.79",0,0,INE_EES_28201
3,Andalucía,CCAA,2011,Hombres,25_34,"18,903.23",0,0,INE_EES_28201
4,Andalucía,CCAA,2011,Mujeres,25_34,"15,194.94",0,0,INE_EES_28201
5,Andalucía,CCAA,2011,Total,25_34,"16,969.88",0,0,INE_EES_28201
6,Andalucía,CCAA,2012,Hombres,25_34,"18,590.66",0,0,INE_EES_28201
7,Andalucía,CCAA,2012,Mujeres,25_34,"14,744.31",0,0,INE_EES_28201
8,Andalucía,CCAA,2012,Total,25_34,"16,476.61",0,0,INE_EES_28201
9,Andalucía,CCAA,2013,Hombres,25_34,"17,277.45",0,0,INE_EES_28201


In [170]:
# ============================================================
# 3.6. COMPROBAR DUPLICADOS
# La combinación territorio + año + sexo + grupo de edad
# debe identificar una única observación.
# ============================================================

duplicates = df_salary[
    df_salary.duplicated(
        subset=["territory_name", "year", "sex", "age_group"],
        keep=False
    )
].copy()

print("Número de filas duplicadas:", len(duplicates))
display(duplicates.head(10))

Número de filas duplicadas: 0


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source


In [171]:
# ============================================================
# 3.7. VALIDACIÓN DE CALIDAD DEL DATO
# Comprobamos:
# - negativos remanentes
# - faltantes
# - flags
# ============================================================

print("Nulos en salary_annual_eur:", int(df_salary["salary_annual_eur"].isna().sum()))
print("Missing flag = 1:", int(df_salary["salary_missing_flag"].sum()))
print("Low sample flag = 1:", int(df_salary["salary_low_sample_flag"].sum()))
print("Salarios negativos tras tratamiento:", int((df_salary["salary_annual_eur"] < 0).sum()))

Nulos en salary_annual_eur: 163
Missing flag = 1: 163
Low sample flag = 1: 607
Salarios negativos tras tratamiento: 0


In [172]:
# ============================================================
# 3.8. RESUMEN DESCRIPTIVO BÁSICO
# Este bloque ayuda a documentar la tabla observada antes de
# imputar faltantes.
# ============================================================

print("Resumen de faltantes por grupo de edad:")
display(
    df_salary.groupby("age_group", as_index=False)["salary_missing_flag"].sum()
)

print("\nResumen de baja muestra por grupo de edad:")
display(
    df_salary.groupby("age_group", as_index=False)["salary_low_sample_flag"].sum()
)

print("\nResumen de faltantes por sexo:")
display(
    df_salary.groupby("sex", as_index=False)["salary_missing_flag"].sum()
)

Resumen de faltantes por grupo de edad:


,age_group,salary_missing_flag
0,25_34,0
1,35_44,0
2,45_54,0
3,55_plus,0
4,all_ages,0
5,lt_25,163



Resumen de baja muestra por grupo de edad:


,age_group,salary_low_sample_flag
0,25_34,48
1,35_44,0
2,45_54,4
3,55_plus,82
4,all_ages,0
5,lt_25,473



Resumen de faltantes por sexo:


,sex,salary_missing_flag
0,Hombres,44
1,Mujeres,112
2,Total,7


In [173]:
# ============================================================
# 3.9. GUARDAR LA VERSIÓN OBSERVADA
# Esta salida conserva la fuente ya limpia y estructurada,
# pero todavía sin imputación de faltantes.
# ============================================================

Path(output_observed).parent.mkdir(parents=True, exist_ok=True)

df_salary.to_csv(output_observed, index=False, encoding="utf-8-sig")

print("Archivo guardado en:")
print(output_observed)

Archivo guardado en:
/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_observed.csv


In [174]:
# ============================================================
# 3.10. RELEER salary_observed.csv PARA COMPROBARLO
# ============================================================

df_salary_check = pd.read_csv(output_observed)

print("Dimensiones del CSV guardado:", df_salary_check.shape)
display(df_salary_check.head(10))

Dimensiones del CSV guardado: (4536, 9)


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source
0,Andalucía,CCAA,2010,Hombres,25_34,"19,574.29",0,0,INE_EES_28201
1,Andalucía,CCAA,2010,Mujeres,25_34,"15,130.18",0,0,INE_EES_28201
2,Andalucía,CCAA,2010,Total,25_34,"17,299.79",0,0,INE_EES_28201
3,Andalucía,CCAA,2011,Hombres,25_34,"18,903.23",0,0,INE_EES_28201
4,Andalucía,CCAA,2011,Mujeres,25_34,"15,194.94",0,0,INE_EES_28201
5,Andalucía,CCAA,2011,Total,25_34,"16,969.88",0,0,INE_EES_28201
6,Andalucía,CCAA,2012,Hombres,25_34,"18,590.66",0,0,INE_EES_28201
7,Andalucía,CCAA,2012,Mujeres,25_34,"14,744.31",0,0,INE_EES_28201
8,Andalucía,CCAA,2012,Total,25_34,"16,476.61",0,0,INE_EES_28201
9,Andalucía,CCAA,2013,Hombres,25_34,"17,277.45",0,0,INE_EES_28201


## **Conclusión del bloque 3**

Al finalizar este bloque se obtiene la primera salida oficial del notebook: `salary_observed.csv`.

Esta versión:
- conserva únicamente la información observada,
- ya ha sido limpiada y transformada,
- incorpora la estructura analítica final,
- y mantiene trazabilidad suficiente para diferenciar faltantes y observaciones de baja robustez muestral.

El siguiente paso será diagnosticar e imputar los valores faltantes para construir `salary_completed.csv`.

# **BLOQUE 4 — Diagnóstico e imputación de faltantes**


Una vez construida la versión observada de la fuente salarial, el siguiente paso consiste en analizar de forma explícita los valores faltantes y aplicar un criterio de imputación coherente con la naturaleza temporal de la serie.

El criterio adoptado en este notebook es el siguiente:

- **Interpolación lineal** para huecos cortos e internos, es decir, cuando faltan uno o dos años y existen observaciones válidas antes y después del hueco.
- **Tasa de crecimiento nacional** para huecos más largos o cuando no es posible interpolar de forma fiable.

Este enfoque permite completar la serie manteniendo la trazabilidad del proceso y diferenciando claramente entre:
- datos observados,
- datos imputados,
- y método de imputación utilizado.

In [175]:
# ============================================================
# BLOQUE 4. DIAGNÓSTICO E IMPUTACIÓN DE FALTANTES
# 4.1. MAPA EXACTO DE VALORES FALTANTES
# Antes de imputar, identificamos con precisión dónde faltan
# datos en la serie salarial observada.
# ============================================================

na_map = (
    df_salary[df_salary["salary_annual_eur"].isna()]
    .sort_values(["territory_name", "sex", "age_group", "year"])
    [["territory_name", "territory_type", "sex", "age_group", "year",
      "salary_annual_eur", "salary_missing_flag", "salary_low_sample_flag"]]
    .reset_index(drop=True)
)

print("Número total de NAs en salary_annual_eur:", len(na_map))
display(na_map.head(50))

print("\nResumen de NAs por territorio, sexo y grupo de edad:")
display(
    na_map.groupby(["territory_name", "sex", "age_group"], as_index=False)
    .agg(
        n_missing_years=("year", "count"),
        years_missing=("year", lambda x: sorted(list(x)))
    )
    .sort_values(["territory_name", "sex", "age_group"])
)

Número total de NAs en salary_annual_eur: 163


,territory_name,territory_type,sex,age_group,year,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
0,Aragón,CCAA,Hombres,lt_25,2016,NaN,1,0
1,Aragón,CCAA,Mujeres,lt_25,2012,NaN,1,0
2,Aragón,CCAA,Mujeres,lt_25,2013,NaN,1,0
3,Aragón,CCAA,Mujeres,lt_25,2015,NaN,1,0
4,Aragón,CCAA,Mujeres,lt_25,2016,NaN,1,0
5,Aragón,CCAA,Mujeres,lt_25,2017,NaN,1,0
6,Aragón,CCAA,Mujeres,lt_25,2019,NaN,1,0
7,Aragón,CCAA,Mujeres,lt_25,2020,NaN,1,0
8,Aragón,CCAA,Mujeres,lt_25,2021,NaN,1,0
9,Asturias,CCAA,Hombres,lt_25,2012,NaN,1,0



Resumen de NAs por territorio, sexo y grupo de edad:


,territory_name,sex,age_group,n_missing_years,years_missing
0,Aragón,Hombres,lt_25,1,[2016]
1,Aragón,Mujeres,lt_25,8,"[2012, 2013, 2015, 2016, 2017, 2019, 2020, 2021]"
2,Asturias,Hombres,lt_25,10,"[2012, 2013, 2014, 2015, 2016, 2017, 2018, 201..."
3,Asturias,Mujeres,lt_25,13,"[2011, 2012, 2013, 2014, 2015, 2016, 2017, 201..."
4,Asturias,Total,lt_25,2,"[2016, 2017]"
5,Baleares,Hombres,lt_25,1,[2013]
6,Baleares,Mujeres,lt_25,2,"[2013, 2016]"
7,Canarias,Hombres,lt_25,1,[2016]
8,Canarias,Mujeres,lt_25,5,"[2013, 2015, 2017, 2020, 2021]"
9,Cantabria,Hombres,lt_25,10,"[2012, 2013, 2014, 2015, 2016, 2017, 2018, 201..."


## **Preparación de la versión completada**

La imputación no se aplicará sobre `salary_observed.csv`, ya que esa versión debe conservarse como reflejo fiel de la fuente observada.

Por ello, se crea una nueva tabla (`df_salary_completed`) sobre la que se aplicarán las imputaciones. Además, se incorporan variables de trazabilidad para distinguir entre:
- observaciones originales,
- observaciones imputadas,
- y método utilizado en cada caso.

In [176]:
# ============================================================
# 4.2. PREPARAR LA TABLA COMPLETADA
# Creamos una copia de la tabla observada y añadimos flags
# de observación e imputación.
# ============================================================

df_salary_completed = df_salary.copy()

# Flag de observación original
df_salary_completed["salary_observed_flag"] = (
    df_salary_completed["salary_annual_eur"].notna()
).astype(int)

# Flag de imputación
df_salary_completed["salary_imputed_flag"] = 0

# Método de imputación utilizado
df_salary_completed["imputation_method"] = pd.NA

# Orden por seguridad
df_salary_completed = df_salary_completed.sort_values(
    ["territory_name", "sex", "age_group", "year"]
).reset_index(drop=True)

display(df_salary_completed.head(10))

,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source,salary_observed_flag,salary_imputed_flag,imputation_method
0,Andalucía,CCAA,2010,Hombres,25_34,"19,574.29",0,0,INE_EES_28201,1,0,<NA>
1,Andalucía,CCAA,2011,Hombres,25_34,"18,903.23",0,0,INE_EES_28201,1,0,<NA>
2,Andalucía,CCAA,2012,Hombres,25_34,"18,590.66",0,0,INE_EES_28201,1,0,<NA>
3,Andalucía,CCAA,2013,Hombres,25_34,"17,277.45",0,0,INE_EES_28201,1,0,<NA>
4,Andalucía,CCAA,2014,Hombres,25_34,"17,586.35",0,0,INE_EES_28201,1,0,<NA>
5,Andalucía,CCAA,2015,Hombres,25_34,"17,714.80",0,0,INE_EES_28201,1,0,<NA>
6,Andalucía,CCAA,2016,Hombres,25_34,"17,555.24",0,0,INE_EES_28201,1,0,<NA>
7,Andalucía,CCAA,2017,Hombres,25_34,"17,724.05",0,0,INE_EES_28201,1,0,<NA>
8,Andalucía,CCAA,2018,Hombres,25_34,"18,269.69",0,0,INE_EES_28201,1,0,<NA>
9,Andalucía,CCAA,2019,Hombres,25_34,"17,978.73",0,0,INE_EES_28201,1,0,<NA>


## **Construcción de la referencia nacional**

Para imputar los huecos largos se utilizará como apoyo la evolución de la serie nacional (`España`) para el mismo sexo y grupo de edad.

La idea es sencilla:
- si no puede interpolarse un hueco con seguridad,
- se utiliza la tasa de crecimiento observada a nivel nacional como patrón de evolución.

In [177]:
# ============================================================
# 4.3. CONSTRUIR REFERENCIA NACIONAL
# Calculamos la tasa de crecimiento nacional por sexo y grupo
# de edad, que servirá para imputar huecos largos.
# ============================================================

national_ref = (
    df_salary_completed[df_salary_completed["territory_name"] == "España"]
    [["sex", "age_group", "year", "salary_annual_eur"]]
    .sort_values(["sex", "age_group", "year"])
    .copy()
)

national_ref["national_growth"] = (
    national_ref.groupby(["sex", "age_group"])["salary_annual_eur"]
    .pct_change()
)

display(national_ref.head(20))

,sex,age_group,year,salary_annual_eur,national_growth
2520,Hombres,25_34,2010,"21,542.06",NaN
2521,Hombres,25_34,2011,"21,189.68",-0.02
2522,Hombres,25_34,2012,"20,708.77",-0.02
2523,Hombres,25_34,2013,"19,766.54",-0.05
2524,Hombres,25_34,2014,"20,278.69",0.03
2525,Hombres,25_34,2015,"19,910.23",-0.02
2526,Hombres,25_34,2016,"19,501.28",-0.02
2527,Hombres,25_34,2017,"20,057.08",0.03
2528,Hombres,25_34,2018,"21,101.42",0.05
2529,Hombres,25_34,2019,"21,026.90",-0.00


## **Regla de imputación**

La imputación se aplicará grupo a grupo, entendiendo cada grupo como una combinación única de:
- territorio,
- sexo,
- grupo de edad.

Dentro de cada grupo:
- los huecos internos de **uno o dos años** se imputarán por **interpolación lineal**,
- los huecos más largos se imputarán mediante **crecimiento nacional**.

Este criterio permite tratar de forma distinta los casos simples y los casos estructuralmente más complejos.

In [178]:
# ============================================================
# 4.4. FUNCIÓN DE IMPUTACIÓN POR GRUPO
# Reglas:
# - huecos cortos internos (<= 2 años): interpolación lineal
# - huecos largos o no interpolables: crecimiento nacional
# ============================================================

def fill_salary_group(group, national_ref):
    """
    Imputa la serie salarial dentro de un grupo definido por:
    territory_name + sex + age_group

    Reglas:
    - Huecos cortos internos (<=2 años y con dato antes y después):
      interpolación lineal
    - Resto de huecos:
      crecimiento nacional
    """
    g = group.sort_values("year").copy().reset_index(drop=True)

    # Serie salarial auxiliar
    s = g["salary_annual_eur"].copy()

    # Máscara de faltantes
    is_missing = s.isna()

    # Si no hay nulos, no hacemos nada
    if not is_missing.any():
        return g

    # --------------------------------------------------------
    # Detectar bloques consecutivos de nulos
    # --------------------------------------------------------
    runs = []
    start = None

    for i, missing in enumerate(is_missing):
        if missing and start is None:
            start = i
        elif (not missing) and start is not None:
            runs.append((start, i - 1))
            start = None

    if start is not None:
        runs.append((start, len(g) - 1))

    sex = g.loc[0, "sex"]
    age_group = g.loc[0, "age_group"]

    # --------------------------------------------------------
    # Imputar bloque por bloque
    # --------------------------------------------------------
    for run_start, run_end in runs:
        run_len = run_end - run_start + 1

        prev_pos = run_start - 1 if run_start > 0 else None
        next_pos = run_end + 1 if run_end < len(g) - 1 else None

        prev_ok = prev_pos is not None and pd.notna(g.loc[prev_pos, "salary_annual_eur"])
        next_ok = next_pos is not None and pd.notna(g.loc[next_pos, "salary_annual_eur"])

        # ----------------------------------------------------
        # CASO A. Hueco corto interno -> interpolación lineal
        # ----------------------------------------------------
        if run_len <= 2 and prev_ok and next_ok:
            interpolated = g["salary_annual_eur"].interpolate(method="linear")

            for pos in range(run_start, run_end + 1):
                g.loc[pos, "salary_annual_eur"] = interpolated.loc[pos]
                g.loc[pos, "salary_imputed_flag"] = 1
                g.loc[pos, "imputation_method"] = "linear_interpolation"

            continue

        # ----------------------------------------------------
        # CASO B. Hueco largo -> crecimiento nacional hacia delante
        # ----------------------------------------------------
        if prev_ok:
            prev_value = g.loc[prev_pos, "salary_annual_eur"]
            success = True
            tmp_values = {}

            for pos in range(run_start, run_end + 1):
                year = int(g.loc[pos, "year"])

                nat_growth = national_ref.loc[
                    (national_ref["sex"] == sex) &
                    (national_ref["age_group"] == age_group) &
                    (national_ref["year"] == year),
                    "national_growth"
                ]

                if nat_growth.empty or pd.isna(nat_growth.iloc[0]):
                    success = False
                    break

                prev_value = prev_value * (1 + nat_growth.iloc[0])
                tmp_values[pos] = prev_value

            if success:
                for pos, value in tmp_values.items():
                    g.loc[pos, "salary_annual_eur"] = value
                    g.loc[pos, "salary_imputed_flag"] = 1
                    g.loc[pos, "imputation_method"] = "national_growth_forward"

                continue

        # ----------------------------------------------------
        # CASO C. Si no se puede hacia delante, backcast con
        # crecimiento nacional usando el valor siguiente
        # ----------------------------------------------------
        if next_ok:
            next_value = g.loc[next_pos, "salary_annual_eur"]
            success = True
            tmp_values = {}

            for pos in range(run_end, run_start - 1, -1):
                year = int(g.loc[pos, "year"])

                nat_growth_next = national_ref.loc[
                    (national_ref["sex"] == sex) &
                    (national_ref["age_group"] == age_group) &
                    (national_ref["year"] == year + 1),
                    "national_growth"
                ]

                if nat_growth_next.empty or pd.isna(nat_growth_next.iloc[0]) or (1 + nat_growth_next.iloc[0]) == 0:
                    success = False
                    break

                next_value = next_value / (1 + nat_growth_next.iloc[0])
                tmp_values[pos] = next_value

            if success:
                for pos, value in tmp_values.items():
                    g.loc[pos, "salary_annual_eur"] = value
                    g.loc[pos, "salary_imputed_flag"] = 1
                    g.loc[pos, "imputation_method"] = "national_growth_backward"

                continue

    return g

In [179]:
# ============================================================
# 4.5. APLICAR LA IMPUTACIÓN A TODA LA TABLA
# Aplicamos la función grupo a grupo:
# territory_name + sex + age_group
# ============================================================

df_salary_completed = (
    df_salary_completed
    .groupby(["territory_name", "sex", "age_group"], group_keys=False)
    .apply(lambda g: fill_salary_group(g, national_ref))
    .reset_index(drop=True)
)

display(df_salary_completed.head(20))

/tmp/ipykernel_3150/266777777.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: fill_salary_group(g, national_ref))


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source,salary_observed_flag,salary_imputed_flag,imputation_method
0,Andalucía,CCAA,2010,Hombres,25_34,"19,574.29",0,0,INE_EES_28201,1,0,NaN
1,Andalucía,CCAA,2011,Hombres,25_34,"18,903.23",0,0,INE_EES_28201,1,0,NaN
2,Andalucía,CCAA,2012,Hombres,25_34,"18,590.66",0,0,INE_EES_28201,1,0,NaN
3,Andalucía,CCAA,2013,Hombres,25_34,"17,277.45",0,0,INE_EES_28201,1,0,NaN
4,Andalucía,CCAA,2014,Hombres,25_34,"17,586.35",0,0,INE_EES_28201,1,0,NaN
5,Andalucía,CCAA,2015,Hombres,25_34,"17,714.80",0,0,INE_EES_28201,1,0,NaN
6,Andalucía,CCAA,2016,Hombres,25_34,"17,555.24",0,0,INE_EES_28201,1,0,NaN
7,Andalucía,CCAA,2017,Hombres,25_34,"17,724.05",0,0,INE_EES_28201,1,0,NaN
8,Andalucía,CCAA,2018,Hombres,25_34,"18,269.69",0,0,INE_EES_28201,1,0,NaN
9,Andalucía,CCAA,2019,Hombres,25_34,"17,978.73",0,0,INE_EES_28201,1,0,NaN


## **Validación post-imputación**

Una vez imputados los valores faltantes, se comprueba:
- cuántos nulos había antes,
- cuántos quedan después,
- cuántas filas se han imputado,
- y qué método se ha utilizado en cada caso.

Este paso es imprescindible para garantizar trazabilidad y evitar una imputación opaca.

In [180]:
# ============================================================
# 4.6. VALIDACIÓN POST-IMPUTACIÓN
# ============================================================

print("NAs antes de imputar:", int(df_salary["salary_annual_eur"].isna().sum()))
print("NAs después de imputar:", int(df_salary_completed["salary_annual_eur"].isna().sum()))
print("Filas imputadas:", int(df_salary_completed["salary_imputed_flag"].sum()))

print("\nMétodos utilizados:")
display(
    df_salary_completed["imputation_method"]
    .value_counts(dropna=False)
    .rename_axis("imputation_method")
    .reset_index(name="n_rows")
)

print("\nResumen de imputaciones por territorio, sexo y grupo de edad:")
display(
    df_salary_completed[df_salary_completed["salary_imputed_flag"] == 1]
    .groupby(["territory_name", "sex", "age_group"], as_index=False)
    .agg(
        n_imputed=("year", "count"),
        years_imputed=("year", lambda x: sorted(list(x))),
        methods=("imputation_method", lambda x: sorted(list(set(x))))
    )
    .sort_values(["territory_name", "sex", "age_group"])
)

NAs antes de imputar: 163
NAs después de imputar: 0
Filas imputadas: 163

Métodos utilizados:


,imputation_method,n_rows
0,NaN,4158
1,<NA>,215
2,national_growth_forward,129
3,linear_interpolation,34



Resumen de imputaciones por territorio, sexo y grupo de edad:


,territory_name,sex,age_group,n_imputed,years_imputed,methods
0,Aragón,Hombres,lt_25,1,[2016],[linear_interpolation]
1,Aragón,Mujeres,lt_25,8,"[2012, 2013, 2015, 2016, 2017, 2019, 2020, 2021]","[linear_interpolation, national_growth_forward]"
2,Asturias,Hombres,lt_25,10,"[2012, 2013, 2014, 2015, 2016, 2017, 2018, 201...",[national_growth_forward]
3,Asturias,Mujeres,lt_25,13,"[2011, 2012, 2013, 2014, 2015, 2016, 2017, 201...",[national_growth_forward]
4,Asturias,Total,lt_25,2,"[2016, 2017]",[linear_interpolation]
5,Baleares,Hombres,lt_25,1,[2013],[linear_interpolation]
6,Baleares,Mujeres,lt_25,2,"[2013, 2016]",[linear_interpolation]
7,Canarias,Hombres,lt_25,1,[2016],[linear_interpolation]
8,Canarias,Mujeres,lt_25,5,"[2013, 2015, 2017, 2020, 2021]",[linear_interpolation]
9,Cantabria,Hombres,lt_25,10,"[2012, 2013, 2014, 2015, 2016, 2017, 2018, 201...",[national_growth_forward]


In [181]:
# ============================================================
# 4.7. GUARDAR LA VERSIÓN COMPLETADA
# Esta versión ya incorpora imputación de faltantes, pero
# todavía no incluye las estimaciones 2024–2025.
# ============================================================

Path(output_completed).parent.mkdir(parents=True, exist_ok=True)

df_salary_completed.to_csv(output_completed, index=False, encoding="utf-8-sig")

print("Archivo guardado en:")
print(output_completed)

Archivo guardado en:
/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_completed.csv


In [182]:
# ============================================================
# 4.8. RELEER salary_completed.csv PARA COMPROBARLO
# ============================================================

df_salary_completed_check = pd.read_csv(output_completed)

print("Dimensiones del CSV completado:", df_salary_completed_check.shape)
display(df_salary_completed_check.head(10))

Dimensiones del CSV completado: (4536, 12)


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source,salary_observed_flag,salary_imputed_flag,imputation_method
0,Andalucía,CCAA,2010,Hombres,25_34,"19,574.29",0,0,INE_EES_28201,1,0,NaN
1,Andalucía,CCAA,2011,Hombres,25_34,"18,903.23",0,0,INE_EES_28201,1,0,NaN
2,Andalucía,CCAA,2012,Hombres,25_34,"18,590.66",0,0,INE_EES_28201,1,0,NaN
3,Andalucía,CCAA,2013,Hombres,25_34,"17,277.45",0,0,INE_EES_28201,1,0,NaN
4,Andalucía,CCAA,2014,Hombres,25_34,"17,586.35",0,0,INE_EES_28201,1,0,NaN
5,Andalucía,CCAA,2015,Hombres,25_34,"17,714.80",0,0,INE_EES_28201,1,0,NaN
6,Andalucía,CCAA,2016,Hombres,25_34,"17,555.24",0,0,INE_EES_28201,1,0,NaN
7,Andalucía,CCAA,2017,Hombres,25_34,"17,724.05",0,0,INE_EES_28201,1,0,NaN
8,Andalucía,CCAA,2018,Hombres,25_34,"18,269.69",0,0,INE_EES_28201,1,0,NaN
9,Andalucía,CCAA,2019,Hombres,25_34,"17,978.73",0,0,INE_EES_28201,1,0,NaN


## **Conclusión del bloque 4**

Al finalizar este bloque se dispone de una segunda salida oficial del notebook: `salary_completed.csv`.

Esta versión:
- conserva la estructura analítica final de la fuente,
- incorpora la imputación documentada de faltantes,
- diferencia claramente entre observaciones originales e imputadas,
- y mantiene trazabilidad sobre el método aplicado en cada caso.

El siguiente paso será armonizar la serie con el periodo final del estudio mediante la estimación de 2024 y 2025.

# **BLOQUE 5 — Estimación 2024–2025 y cierre**

El periodo final del trabajo se extiende hasta 2025, pero la fuente salarial observada del INE solo llega hasta 2023. Por ello, en este bloque se generan dos años adicionales de forma documentada y reproducible.

Para minimizar la arbitrariedad, se utilizan **proxies oficiales del INE**:

- **2024**: estimado aplicando una tasa de crecimiento del **3,8%** sobre 2023
- **2025**: estimado aplicando una tasa de crecimiento del **3,6%** sobre 2024

Estas observaciones no proceden directamente de la Encuesta de Estructura Salarial, sino que se incorporan como **estimaciones auxiliares** para armonizar el panel final 2010–2025.

In [183]:
# ============================================================
# BLOQUE 5. ESTIMACIÓN 2024–2025 Y CIERRE
# 5.1. PREPARAR LA BASE FINAL
# Partimos de la versión completada de salarios, que ya incluye
# la imputación de faltantes en el periodo observado 2010–2023.
# ============================================================

df_salary_final = df_salary_completed.copy()

print("Dimensiones base antes de estimar 2024–2025:", df_salary_final.shape)
print("Rango temporal actual:", df_salary_final["year"].min(), "-", df_salary_final["year"].max())

Dimensiones base antes de estimar 2024–2025: (4536, 12)
Rango temporal actual: 2010 - 2023


In [184]:
# ============================================================
# 5.2. ESTIMAR 2024
# Regla metodológica:
# salario_2024 = salario_2023 * 1.038
# Proxy oficial utilizado:
# Encuesta Anual de Coste Laboral (INE), variación 2024 = +3,8%
# ============================================================

df_2024 = df_salary_final[df_salary_final["year"] == 2023].copy()

df_2024["year"] = 2024
df_2024["salary_annual_eur"] = df_2024["salary_annual_eur"] * 1.038

# Flags de trazabilidad
df_2024["salary_observed_flag"] = 0
df_2024["salary_imputed_flag"] = 1
df_2024["imputation_method"] = "growth_estimation_2024_3_8pct"

print("Filas generadas para 2024:", len(df_2024))
display(df_2024.head(10))

Filas generadas para 2024: 324


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source,salary_observed_flag,salary_imputed_flag,imputation_method
13,Andalucía,CCAA,2024,Hombres,25_34,"22,035.86",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
27,Andalucía,CCAA,2024,Hombres,35_44,"26,988.45",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
41,Andalucía,CCAA,2024,Hombres,45_54,"31,870.95",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
55,Andalucía,CCAA,2024,Hombres,55_plus,"32,376.26",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
69,Andalucía,CCAA,2024,Hombres,all_ages,"28,114.75",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
83,Andalucía,CCAA,2024,Hombres,lt_25,"15,041.90",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
97,Andalucía,CCAA,2024,Mujeres,25_34,"19,465.12",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
111,Andalucía,CCAA,2024,Mujeres,35_44,"22,178.20",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
125,Andalucía,CCAA,2024,Mujeres,45_54,"25,807.39",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
139,Andalucía,CCAA,2024,Mujeres,55_plus,"28,640.98",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct


In [185]:
# ============================================================
# # ============================================================
# 5.3. ESTIMAR 2025
# Regla metodológica:
# salario_2025 = salario_2024 * 1.036
# Proxy oficial utilizado:
# ETCL 4T 2025 (INE), crecimiento del coste salarial = +3,6%
# ============================================================

df_2025 = df_2024.copy()

df_2025["year"] = 2025
df_2025["salary_annual_eur"] = df_2025["salary_annual_eur"] * 1.036

# Flags de trazabilidad
df_2025["salary_observed_flag"] = 0
df_2025["salary_imputed_flag"] = 1
df_2025["imputation_method"] = "growth_estimation_2025_3_6pct"

print("Filas generadas para 2025:", len(df_2025))
display(df_2025.head(10))

Filas generadas para 2025: 324


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source,salary_observed_flag,salary_imputed_flag,imputation_method
13,Andalucía,CCAA,2025,Hombres,25_34,"22,829.15",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct
27,Andalucía,CCAA,2025,Hombres,35_44,"27,960.03",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct
41,Andalucía,CCAA,2025,Hombres,45_54,"33,018.30",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct
55,Andalucía,CCAA,2025,Hombres,55_plus,"33,541.80",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct
69,Andalucía,CCAA,2025,Hombres,all_ages,"29,126.88",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct
83,Andalucía,CCAA,2025,Hombres,lt_25,"15,583.41",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct
97,Andalucía,CCAA,2025,Mujeres,25_34,"20,165.86",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct
111,Andalucía,CCAA,2025,Mujeres,35_44,"22,976.61",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct
125,Andalucía,CCAA,2025,Mujeres,45_54,"26,736.46",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct
139,Andalucía,CCAA,2025,Mujeres,55_plus,"29,672.06",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct


## **Integración de la serie final**

Una vez construidas las estimaciones de 2024 y 2025, se integran con la serie completada 2010–2023 para obtener una única tabla salarial armonizada 2010–2025.

Es importante señalar que:
- 2010–2023 corresponden a observaciones de la fuente salarial tratada,
- 2024 y 2025 son estimaciones auxiliares basadas en proxies oficiales del INE,
- y todas las filas estimadas quedan marcadas con trazabilidad específica.

In [186]:
# ============================================================
# 5.4. INTEGRAR LA SERIE FINAL 2010–2025
# ============================================================

df_salary_final = pd.concat(
    [df_salary_final, df_2024, df_2025],
    ignore_index=True
)

df_salary_final = df_salary_final.sort_values(
    by=["territory_type", "territory_name", "age_group", "year", "sex"]
).reset_index(drop=True)

print("Dimensiones tras añadir 2024 y 2025:", df_salary_final.shape)
print("Rango temporal final:", df_salary_final["year"].min(), "-", df_salary_final["year"].max())

display(df_salary_final.head(20))
display(df_salary_final.tail(20))

Dimensiones tras añadir 2024 y 2025: (5184, 12)
Rango temporal final: 2010 - 2025


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source,salary_observed_flag,salary_imputed_flag,imputation_method
0,Andalucía,CCAA,2010,Hombres,25_34,"19,574.29",0,0,INE_EES_28201,1,0,NaN
1,Andalucía,CCAA,2010,Mujeres,25_34,"15,130.18",0,0,INE_EES_28201,1,0,NaN
2,Andalucía,CCAA,2010,Total,25_34,"17,299.79",0,0,INE_EES_28201,1,0,NaN
3,Andalucía,CCAA,2011,Hombres,25_34,"18,903.23",0,0,INE_EES_28201,1,0,NaN
4,Andalucía,CCAA,2011,Mujeres,25_34,"15,194.94",0,0,INE_EES_28201,1,0,NaN
5,Andalucía,CCAA,2011,Total,25_34,"16,969.88",0,0,INE_EES_28201,1,0,NaN
6,Andalucía,CCAA,2012,Hombres,25_34,"18,590.66",0,0,INE_EES_28201,1,0,NaN
7,Andalucía,CCAA,2012,Mujeres,25_34,"14,744.31",0,0,INE_EES_28201,1,0,NaN
8,Andalucía,CCAA,2012,Total,25_34,"16,476.61",0,0,INE_EES_28201,1,0,NaN
9,Andalucía,CCAA,2013,Hombres,25_34,"17,277.45",0,0,INE_EES_28201,1,0,NaN


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source,salary_observed_flag,salary_imputed_flag,imputation_method
5164,España,España,2019,Mujeres,lt_25,"11,103.13",0,0,INE_EES_28201,1,0,NaN
5165,España,España,2019,Total,lt_25,"12,270.40",0,0,INE_EES_28201,1,0,NaN
5166,España,España,2020,Hombres,lt_25,"13,783.60",0,0,INE_EES_28201,1,0,NaN
5167,España,España,2020,Mujeres,lt_25,"11,333.14",0,0,INE_EES_28201,1,0,NaN
5168,España,España,2020,Total,lt_25,"12,693.72",0,0,INE_EES_28201,1,0,NaN
5169,España,España,2021,Hombres,lt_25,"14,056.13",0,0,INE_EES_28201,1,0,NaN
5170,España,España,2021,Mujeres,lt_25,"11,498.86",0,0,INE_EES_28201,1,0,NaN
5171,España,España,2021,Total,lt_25,"12,930.13",0,0,INE_EES_28201,1,0,NaN
5172,España,España,2022,Hombres,lt_25,"15,970.98",0,0,INE_EES_28201,1,0,NaN
5173,España,España,2022,Mujeres,lt_25,"13,212.09",0,0,INE_EES_28201,1,0,NaN


In [187]:
# ============================================================
# 5.5. VALIDACIÓN DE LA SERIE FINAL
# Comprobamos:
# - cobertura temporal 2010–2025
# - cobertura territorial
# - no duplicados
# - ausencia de salarios negativos
# ============================================================

print("Número de filas:", len(df_salary_final))
print("Territorios únicos:", df_salary_final["territory_name"].nunique())
print("Sexos únicos:", sorted(df_salary_final["sex"].unique()))
print("Grupos de edad:", sorted(df_salary_final["age_group"].unique()))
print("Rango temporal:", df_salary_final["year"].min(), "-", df_salary_final["year"].max())
print("Salarios negativos:", int((df_salary_final["salary_annual_eur"] < 0).sum()))
print("Nulos en salary_annual_eur:", int(df_salary_final["salary_annual_eur"].isna().sum()))

duplicates_final = df_salary_final[
    df_salary_final.duplicated(
        subset=["territory_name", "year", "sex", "age_group"],
        keep=False
    )
]

print("Filas duplicadas:", len(duplicates_final))
display(duplicates_final.head(10))

Número de filas: 5184
Territorios únicos: 18
Sexos únicos: ['Hombres', 'Mujeres', 'Total']
Grupos de edad: ['25_34', '35_44', '45_54', '55_plus', 'all_ages', 'lt_25']
Rango temporal: 2010 - 2025
Salarios negativos: 0
Nulos en salary_annual_eur: 0
Filas duplicadas: 0


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source,salary_observed_flag,salary_imputed_flag,imputation_method


In [188]:
# ============================================================
# 5.6. RESUMEN DE TRAZABILIDAD
# Mostramos cuántas filas son:
# - observadas
# - imputadas
# - estimadas para 2024–2025
# ============================================================

print("Resumen de métodos utilizados:")
display(
    df_salary_final["imputation_method"]
    .value_counts(dropna=False)
    .rename_axis("imputation_method")
    .reset_index(name="n_rows")
)

print("\nResumen por año:")
display(
    df_salary_final.groupby("year", as_index=False).agg(
        n_rows=("salary_annual_eur", "count"),
        n_imputed=("salary_imputed_flag", "sum")
    )
)

Resumen de métodos utilizados:


,imputation_method,n_rows
0,NaN,4158
1,growth_estimation_2024_3_8pct,324
2,growth_estimation_2025_3_6pct,324
3,<NA>,215
4,national_growth_forward,129
5,linear_interpolation,34



Resumen por año:


,year,n_rows,n_imputed
0,2010,324,0
1,2011,324,4
2,2012,324,12
3,2013,324,21
4,2014,324,12
5,2015,324,18
6,2016,324,22
7,2017,324,18
8,2018,324,13
9,2019,324,8


In [189]:
# ============================================================
# 5.7. GUARDAR LA SERIE FINAL
# Guardamos la versión salarial final armonizada 2010–2025.
# ============================================================

Path(output_final).parent.mkdir(parents=True, exist_ok=True)

df_salary_final.to_csv(output_final, index=False, encoding="utf-8-sig")

print("Archivo guardado en:")
print(output_final)

Archivo guardado en:
/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_final.csv


In [190]:
# ============================================================
# 5.8. RELEER salary_final.csv PARA COMPROBARLO
# ============================================================

df_salary_final_check = pd.read_csv(output_final)

print("Dimensiones del CSV final:", df_salary_final_check.shape)
display(df_salary_final_check.head(10))
display(df_salary_final_check.tail(10))

Dimensiones del CSV final: (5184, 12)


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source,salary_observed_flag,salary_imputed_flag,imputation_method
0,Andalucía,CCAA,2010,Hombres,25_34,"19,574.29",0,0,INE_EES_28201,1,0,NaN
1,Andalucía,CCAA,2010,Mujeres,25_34,"15,130.18",0,0,INE_EES_28201,1,0,NaN
2,Andalucía,CCAA,2010,Total,25_34,"17,299.79",0,0,INE_EES_28201,1,0,NaN
3,Andalucía,CCAA,2011,Hombres,25_34,"18,903.23",0,0,INE_EES_28201,1,0,NaN
4,Andalucía,CCAA,2011,Mujeres,25_34,"15,194.94",0,0,INE_EES_28201,1,0,NaN
5,Andalucía,CCAA,2011,Total,25_34,"16,969.88",0,0,INE_EES_28201,1,0,NaN
6,Andalucía,CCAA,2012,Hombres,25_34,"18,590.66",0,0,INE_EES_28201,1,0,NaN
7,Andalucía,CCAA,2012,Mujeres,25_34,"14,744.31",0,0,INE_EES_28201,1,0,NaN
8,Andalucía,CCAA,2012,Total,25_34,"16,476.61",0,0,INE_EES_28201,1,0,NaN
9,Andalucía,CCAA,2013,Hombres,25_34,"17,277.45",0,0,INE_EES_28201,1,0,NaN


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source,salary_observed_flag,salary_imputed_flag,imputation_method
5174,España,España,2022,Total,lt_25,"14,700.90",0,0,INE_EES_28201,1,0,NaN
5175,España,España,2023,Hombres,lt_25,"16,191.49",0,0,INE_EES_28201,1,0,NaN
5176,España,España,2023,Mujeres,lt_25,"13,409.09",0,0,INE_EES_28201,1,0,NaN
5177,España,España,2023,Total,lt_25,"14,944.35",0,0,INE_EES_28201,1,0,NaN
5178,España,España,2024,Hombres,lt_25,"16,806.77",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
5179,España,España,2024,Mujeres,lt_25,"13,918.64",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
5180,España,España,2024,Total,lt_25,"15,512.24",0,0,INE_EES_28201,0,1,growth_estimation_2024_3_8pct
5181,España,España,2025,Hombres,lt_25,"17,411.81",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct
5182,España,España,2025,Mujeres,lt_25,"14,419.71",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct
5183,España,España,2025,Total,lt_25,"16,070.68",0,0,INE_EES_28201,0,1,growth_estimation_2025_3_6pct


## **Conclusión final del notebook**

A partir de la fuente salarial original del INE se han construido tres versiones del dataset:

- **`salary_observed.csv`**: versión observada, limpia y estructurada
- **`salary_completed.csv`**: versión con tratamiento documentado de valores faltantes
- **`salary_final.csv`**: versión armonizada 2010–2025, que incorpora además las estimaciones de 2024 y 2025

Las observaciones de 2024 y 2025 no proceden directamente de la Encuesta de Estructura Salarial, sino que se incorporan como estimaciones auxiliares basadas en proxies oficiales del INE, con el objetivo de mantener un panel temporal homogéneo para el análisis posterior.